# Phasing and estimating unmeasured variants

The genotyping array measures selected positions in the genome. Phasing estimates which alleles travel together on each chromosome copy. Imputation uses these patterns and a panel of sequenced dogs to estimate genotypes at additional positions.

The reference-aligned data from the preceding quality-control chapter provide the starting point. We analyze German Shorthaired Pointers and Labrador Retrievers separately, across all 38 autosomes.

## The dogs and reference panel

The procedure includes 116 German Shorthaired Pointers and 48 Labrador Retrievers. Each dog is analyzed once within its breed. The smaller primary comparison groups are selected from these results afterward, so the same dog has the same imputed genotype in both comparisons. Training outcomes do not enter phasing or imputation.

We use the phased Dog10K reference panel on CanFam4, also called UU_Cfam_GSD_1.0. The panel represents diverse dog populations and related canids. Its breed-specific representation is limited: four German Shorthaired Pointers and six Labrador Retrievers. Accuracy in our dogs therefore needs to be assessed rather than assumed.

## Matching the study markers to the panel

At each marker used as an input, the chromosome, position, reference allele, and alternate allele must agree with the reference panel. CanineHD strand interpretation was handled in the quality-control chapter. Here we check whether each aligned marker is present and compatible with the panel.

We restrict this procedure to reference markers with two ordinary DNA alleles, a unique position, a matching CanFam4 reference base, and complete phased diploid genotypes. Study markers that cannot be matched are recorded separately rather than silently changing their allele interpretation.

## Estimating chromosome copies and additional genotypes

Beagle 5.5 estimates phase and imputes additional genotypes using the reference panel and a canine genetic map. We process one chromosome at a time. The procedure preserves a phased version of the input markers and an imputed version containing additional reference-panel variants.

Estimated allele counts, called dosages, retain information about uncertainty for later analyses. Genotype probabilities are also retained. We check that every originally measured, non-missing genotype is preserved. A successful software run alone does not establish imputation accuracy.

## Results

Phasing and imputation completed across all 38 autosomes in both breeds. Every compatible input site was recovered, and every originally measured, non-missing genotype was preserved. The primary comparison groups were selected from the same imputed genotypes afterward.

| Breed | Dogs analyzed | Compatible measured markers | Observed calls preserved |
| --- | ---: | ---: | ---: |
| GSP | 116 | 155,362 | 18,009,866 |
| LR | 48 | 158,380 | 7,586,470 |

All recorded genotype and index file hashes passed verification. The original CanFam3.1 datasets remain unchanged. The next chapter examines recovery of hidden measurements; completion of inference does not establish that every imputed variant is suitable for association. Biological annotation on ROS_Cfam_1.0 remains a separate step.


## Run the whole-autosome procedure

Use a WSL/Linux project kernel for new inference. The tool environment needs bcftools, Java, liftOver, and NumPy; prepare it once from a WSL terminal with `bash scripts/setup_imputation_wsl.sh`. Public Beagle and map resources can be retrieved with `python scripts/download_imputation_assets.py`.

The cells below use the same configuration and script as the completed study. A completed run is inspected without repeating inference. Set `imputation_folder` to a new directory to reproduce the analysis separately. An unfinished matching run can resume. A fresh whole-genome run downloads a large reference panel and may take considerable time.

In [ ]:
from pathlib import Path
import sys

study_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/notebook_support.py").exists())
sys.path.insert(0, str(study_root / "scripts"))
from notebook_support import read_json, study_path, verify_datasets, run_script

In [ ]:
imputation_folder = study_root / "data/imputation/autosomal-20261003"
settings = read_json(study_root / "config/imputation.json")
print("Configured autosomes:", len(settings["chromosomes"]))
print("Configuration:", study_root / "config/imputation.json")

### Prepare chromosome inputs and perform inference

Preparation splits the aligned genotypes and records a reproducible plan. Execution phases and imputes each breed, creates the primary-group subsets, and performs the predefined hidden-genotype runs. The original measured calls are preserved. The full implementation is in [imputation_workflow.py](../scripts/imputation_workflow.py).

In [ ]:
manifest_file = imputation_folder / "manifest.private.json"
completed = manifest_file.exists() and read_json(manifest_file)["status"] == "complete"
if not completed:
    run_script(study_root, "imputation_workflow.py", "--output", imputation_folder,
               "--prepare-only", linux=True)
    run_script(study_root, "imputation_workflow.py", "--output", imputation_folder,
               "--execute", linux=True)
imputation = read_json(manifest_file)
assert imputation["status"] == "complete"
assert set(imputation["jobs"]) == {f"chr{i}" for i in range(1, 39)}
assert all(job["status"] == "complete" for job in imputation["jobs"].values())
print("Completed:", len(imputation["jobs"]), "autosomes in both breeds.")

### Check recovery of the measured input

These summaries were recorded during execution, after comparison with the aligned source genotypes. They check measured-call preservation; they do not establish the accuracy of every newly imputed variant.

In [ ]:
for breed in ["GSP", "LR"]:
    records = [job["breeds"][breed] for job in imputation["jobs"].values()]
    assert all(r["verification"]["all_input_sites_recovered"] for r in records)
    markers = sum(r["panel_compatibility"]["compatible"] for r in records)
    calls = sum(r["verification"]["observed_calls_preserved"] for r in records)
    print(f"{breed}: {markers:,} compatible measured markers; {calls:,} observed calls preserved.")